# GDELT Events — Drive Backfill

Downloads GDELT 1.0 Events data via free HTTP, applies zone-actor filter, saves daily Parquet files  
to Google Drive with zstd compression. Fully resumable — interrupt at any point and rerun to continue.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
from datetime import date

START_DATE        = date(2022, 1, 1)   # Russia-Ukraine anchor — do not move later
END_DATE          = date.today() - __import__('datetime').timedelta(days=1)  # yesterday
SLEEP_BETWEEN     = 1.5                # seconds between HTTP requests
ZSTD_LEVEL        = 12                 # 1-22; 12 = strong compression, reasonable speed
DRIVE_ROOT        = "/content/drive/MyDrive/gdelt_events_bronze"

In [ ]:
# ── Mount Drive ────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import csv
import io
import json
import time
import zipfile
from datetime import timedelta
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Zone constants ─────────────────────────────────────────────────────────────
ZONE_COUNTRY_CODES = {
    "USD": ["USA"],
    "EUR": [
        "EUR", "AUT", "BEL", "CYP", "DEU", "ESP", "EST", "FIN", "FRA",
        "GRC", "HRV", "IRL", "ITA", "LTU", "LUX", "LVA", "MLT",
        "NLD", "PRT", "SVK", "SVN",
    ],
    "GBP": ["GBR"],
    "JPY": ["JPN"],
    "CHF": ["CHE"],
}
ALL_ZONE_CODES = {code for codes in ZONE_COUNTRY_CODES.values() for code in codes}

# ── GDELT 1.0 — 58 columns, tab-separated, no header ──────────────────────────
ALL_COLUMNS = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode", "QuadClass",
    "GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "AvgTone",
    "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_Lat", "Actor1Geo_Long", "Actor1Geo_FeatureID",
    "Actor2Geo_Type", "Actor2Geo_FullName", "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code", "Actor2Geo_Lat", "Actor2Geo_Long", "Actor2Geo_FeatureID",
    "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_Lat", "ActionGeo_Long", "ActionGeo_FeatureID",
    "DATEADDED", "SOURCEURL",
]
assert len(ALL_COLUMNS) == 58

# Columns we keep in Bronze (zone-filtered; QuadClass kept for EDA, not a filter here)
KEEP_COLUMNS = [
    "GLOBALEVENTID", "SQLDATE",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1Type1Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2Type1Code",
    "EventCode", "EventBaseCode", "EventRootCode", "QuadClass",
    "GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "AvgTone",
    "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor2Geo_FullName", "Actor2Geo_CountryCode",
    "ActionGeo_FullName", "ActionGeo_CountryCode",
    "SOURCEURL",
]

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────────
BASE_URL = "http://data.gdeltproject.org/events/{}.export.CSV.zip"


def parquet_path(d: date) -> Path:
    return Path(DRIVE_ROOT) / str(d.year) / f"{d.month:02d}" / f"{d.strftime('%Y%m%d')}.parquet"


def download_day(d: date) -> pd.DataFrame | None:
    """Download, zone-filter, and return a DataFrame for one day. None on failure."""
    url = BASE_URL.format(d.strftime("%Y%m%d"))
    try:
        resp = requests.get(url, timeout=60)
    except requests.RequestException as exc:
        print(f"  [SKIP] {d} request error: {exc}")
        return None

    if resp.status_code == 404:
        print(f"  [SKIP] {d} 404")
        return None
    if resp.status_code != 200:
        print(f"  [SKIP] {d} HTTP {resp.status_code}")
        return None

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        with zf.open(zf.namelist()[0]) as f:
            df = pd.read_csv(
                f,
                sep="\t",
                header=None,
                names=ALL_COLUMNS,
                dtype=str,
                quoting=csv.QUOTE_NONE,
                on_bad_lines="skip",
                usecols=KEEP_COLUMNS,
            )

    # Zone-actor filter — the only filter applied in Bronze
    mask = (
        df["Actor1CountryCode"].isin(ALL_ZONE_CODES)
        | df["Actor2CountryCode"].isin(ALL_ZONE_CODES)
    )
    filtered = df[mask].copy()
    filtered.reset_index(drop=True, inplace=True)
    return filtered, len(df), len(resp.content)


def save_parquet(df: pd.DataFrame, path: Path) -> int:
    """Write DataFrame as zstd-compressed Parquet. Returns file size in bytes."""
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(
        table,
        str(path),
        compression="zstd",
        compression_level=ZSTD_LEVEL,
        use_dictionary=True,
        write_statistics=False,
    )
    return path.stat().st_size


def drive_usage_gb() -> float:
    """Approximate total size of DRIVE_ROOT in GB."""
    total = sum(p.stat().st_size for p in Path(DRIVE_ROOT).rglob("*.parquet") if p.is_file())
    return total / 1e9

In [ ]:
# ── Resume scan ────────────────────────────────────────────────────────────────
# Build the full date list, check which days are already done.
all_days = []
d = START_DATE
while d <= END_DATE:
    all_days.append(d)
    d += timedelta(days=1)

done  = {d for d in all_days if parquet_path(d).exists()}
todo  = [d for d in all_days if d not in done]

print(f"Date range    : {START_DATE} → {END_DATE}  ({len(all_days)} days)")
print(f"Already done  : {len(done)} days")
print(f"Remaining     : {len(todo)} days")
print(f"Drive used    : {drive_usage_gb():.2f} GB")

In [ ]:
# ── Collect ────────────────────────────────────────────────────────────────────
session_collected = 0
session_skipped   = 0
session_bytes     = 0

for i, d in enumerate(todo, 1):
    result = download_day(d)

    if result is None:
        session_skipped += 1
        # Write empty sentinel so we don't retry 404 days on resume
        empty = pd.DataFrame(columns=KEEP_COLUMNS)
        save_parquet(empty, parquet_path(d))
        time.sleep(SLEEP_BETWEEN)
        continue

    df, raw_rows, zip_bytes = result
    file_bytes = save_parquet(df, parquet_path(d))

    session_collected += 1
    session_bytes     += file_bytes

    print(
        f"[{i:>4}/{len(todo)}] {d}  "
        f"raw={raw_rows:>7,}  filtered={len(df):>6,}  "
        f"zip={zip_bytes/1e6:>5.1f}MB  "
        f"parquet={file_bytes/1e6:>5.2f}MB  "
        f"drive={drive_usage_gb():.2f}GB"
    )
    time.sleep(SLEEP_BETWEEN)

print(f"\nSession done. Collected={session_collected}, skipped={session_skipped}, "
      f"written={session_bytes/1e9:.3f} GB  |  Total drive: {drive_usage_gb():.2f} GB")

In [ ]:
# ── Final status ───────────────────────────────────────────────────────────────
done_now = {d for d in all_days if parquet_path(d).exists()}
remaining = len(all_days) - len(done_now)

print("=" * 52)
print("  GDELT Events Backfill Status")
print("=" * 52)
print(f"  Date range  : {START_DATE} → {END_DATE}")
print(f"  Complete    : {len(done_now)} / {len(all_days)} days")
print(f"  Remaining   : {remaining} days")
print(f"  Drive used  : {drive_usage_gb():.3f} GB")
if len(done_now) > 0 and remaining > 0:
    avg_mb = (drive_usage_gb() * 1e3) / len(done_now)
    print(f"  Projected   : {drive_usage_gb() + (avg_mb * remaining / 1e3):.2f} GB when complete")
print("=" * 52)
if remaining == 0:
    print("  Backfill complete.")